In [3]:
%pip install ipywidgets==8.0.7
%pip install ipynao==0.8.9
%pip install naowidgets==0.1.1
from naowidgets import *
import ipynao

PEPPER_IP = "10.219.8.57"     # address of Pepper's HEAD (press the chest button). Change only this line.

# This Pepper does not run the RoboHearts service on port 8443 that notebooks normally talk to, so a bridge on
# this computer stands in for it: bridge/pepper_bridge.py logs in to Pepper's NAOqi (tcps://PEPPER_IP:9503,
# user and password from bridge/.env, factory default nao / nao) and serves ipynao's protocol on 127.0.0.1:8443.
# The line below starts the bridge, or re-uses the running one, and prints whether Pepper is connected.
# On RoboHearts (JupyterLite) there is no bridge: the notebook then connects straight to PEPPER_IP, as the lessons do.
try:
    from bridge.notebook_helper import connect_address
    CONNECT_TO = connect_address(PEPPER_IP)        # "127.0.0.1" while the bridge runs here, else PEPPER_IP
except ImportError:
    CONNECT_TO = PEPPER_IP

pepper = ipynao.Robot(autoconnect=False)   # without this, ipynao first tries to reach the website instead of the robot
pepper.connect(CONNECT_TO)
pepper


Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Setting up the bridge on this computer (first time only, a few minutes): Python 3.12 + the qi package ...
  Prepared 1 package in 30ms
  Installed 1 package in 2ms
   + qi==3.1.5 (from file:///Users/michaelzhang/Desktop/pepper

Robot()

# Step 1 - connect as in the lessons

Put the address of Pepper's **head** in `PEPPER_IP` above (press the chest button and Pepper says it). This is the only place the address is written: the other steps read it from here.

**On this computer** the cell also starts the *Pepper bridge* (`bridge/pepper_bridge.py`). This Pepper does not run the RoboHearts service that notebooks normally talk to on port 8443, so the bridge logs in to Pepper's NAOqi itself (port 9503, user `nao`, password from `bridge/.env`, factory default `nao`) and offers the same connection on `127.0.0.1:8443`. The printed lines say whether Pepper is connected through it.

The widget must say **Connected to 127.0.0.1** (through the bridge), or **Connected to** your `PEPPER_IP` on RoboHearts. If it says *Failed to 127.0.0.1* although the printed lines say Pepper is connected, the browser has not accepted the bridge's certificate yet: open `https://127.0.0.1:8443/check` in a new tab, click *Advanced*, then *Proceed*, and run this cell again (once per browser session; Step 2 does not need it). For any other *Failed*, run the check cell below: it tells you which cause it is.

The first time, Chrome may ask whether this site may *look for and connect to any device on your local network*: click **Allow**.


## Step 1b - check the connection from this computer

Run the cell below after Step 1. It uses `PEPPER_IP` and `CONNECT_TO` from Step 1 and checks, from this computer, exactly what the notebook needs:

* what Pepper's head answers: NAOqi's own ports **9503 / 9443** (open whenever Pepper is on and finished booting) and the RoboHearts port **8443** (only on robots that run the RoboHearts service; not needed with the bridge);
* the **bridge** on this computer: is it running, is it logged in to Pepper, and does it answer the robot's name over the same web socket that Step 1 and Step 2 use.

It prints **CONNECTED** with the robot's name and whether Pepper's camera, wheels and arms answer (read only: nothing moves), or the cause and the fix (bridge not running, wrong password, Pepper off or wrong address, ...). The check only reads; Pepper does not move or speak. On RoboHearts (JupyterLite) the kernel runs inside the browser and cannot open connections, so there the cell just points at the check in the Step 2 box.


In [4]:
# Checks from this computer whether the notebook can reach Pepper, and says what to do if not.
# Standard library only. Two cases:
#   * through the bridge on this computer (CONNECT_TO == 127.0.0.1): bridge running? logged in to Pepper?
#   * directly (RoboHearts, or no bridge): does Pepper's RoboHearts service answer on port 8443?
import base64, json, os, socket, ssl, struct, sys, time, urllib.request

ROBOT_PORT = 8443            # RoboHearts' web-socket service on Pepper's head; the bridge uses the same port on 127.0.0.1
NAOQI_PORTS = (9503, 9443)   # NAOqi 2.9's own secure ports: open whenever NAOqi is running on Pepper

def _tcp(host, port, timeout=3.0):
    """-> (state, milliseconds). state: open | refused | timeout | unreachable | error:<text>"""
    started = time.time()
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return "open", int((time.time() - started) * 1000)
    except ConnectionRefusedError:
        return "refused", int((time.time() - started) * 1000)
    except socket.timeout:
        return "timeout", int((time.time() - started) * 1000)
    except OSError as e:
        if getattr(e, "errno", None) in (64, 65, 101, 113, 51):   # host down / no route
            return "unreachable", int((time.time() - started) * 1000)
        return "error:" + str(e), int((time.time() - started) * 1000)

def _tls_common_name(host, port, timeout=4.0):
    """Common name of the certificate a TLS port presents (without verifying it)."""
    ctx = ssl.SSLContext(ssl.PROTOCOL_TLS_CLIENT)
    ctx.check_hostname = False; ctx.verify_mode = ssl.CERT_NONE
    try:
        with socket.create_connection((host, port), timeout=timeout) as raw:
            with ctx.wrap_socket(raw, server_hostname=host) as tls:
                der = tls.getpeercert(binary_form=True) or b""
    except (OSError, ssl.SSLError) as e:
        return None, str(e)
    names = []
    i = der.find(b"\x55\x04\x03")           # OID 2.5.4.3 = commonName
    while i != -1 and i + 4 < len(der):
        tag, length = der[i + 3], der[i + 4]
        if tag in (0x0c, 0x13, 0x16) and length < 128:
            names.append(der[i + 5:i + 5 + length].decode("latin-1"))
        i = der.find(b"\x55\x04\x03", i + 3)
    return (names[0] if names else "?"), None

def _ws_robot_name(host, port, timeout=8.0, with_skills=False):
    """Opens wss://host:port like ipynao does and asks ALSystem.robotName(); with_skills also asks the bridge's
    PepperSkills.info() (which of Pepper's camera, motion and speech services answer). -> (name, info, error)"""
    ctx = ssl.SSLContext(ssl.PROTOCOL_TLS_CLIENT)
    ctx.check_hostname = False; ctx.verify_mode = ssl.CERT_NONE
    try:
        raw = socket.create_connection((host, port), timeout=timeout)
        s = ctx.wrap_socket(raw, server_hostname=host)
    except (OSError, ssl.SSLError) as e:
        if not _is_loopback(host):
            return None, None, "secure connection failed: " + str(e)
        try:                                     # the bridge on this computer also speaks plain ws:// (what Step 2 uses)
            s = socket.create_connection((host, port), timeout=timeout)
        except OSError as e2:
            return None, None, "connection failed: " + str(e2)
    try:
        key = base64.b64encode(os.urandom(16)).decode()
        s.sendall(("GET / HTTP/1.1\r\nHost: %s:%d\r\nUpgrade: websocket\r\nConnection: Upgrade\r\n"
                   "Sec-WebSocket-Key: %s\r\nSec-WebSocket-Version: 13\r\n\r\n" % (host, port, key)).encode())
        head = b""
        while b"\r\n\r\n" not in head:
            chunk = s.recv(4096)
            if not chunk:
                return None, None, "the server closed the connection during the web-socket handshake"
            head += chunk
        status = head.split(b"\r\n", 1)[0].decode(errors="replace")
        if " 101 " not in status:
            return None, None, "no web socket at this port (server answered: %s)" % status
        buf = [head.split(b"\r\n\r\n", 1)[1]]

        def send(obj):
            payload = json.dumps(obj).encode()
            mask = os.urandom(4)
            n = len(payload)
            hdr = bytes([0x81]) + (bytes([0x80 | n]) if n < 126 else bytes([0x80 | 126]) + struct.pack(">H", n))
            s.sendall(hdr + mask + bytes(c ^ mask[i % 4] for i, c in enumerate(payload)))

        def recv_exact(n):
            data = buf[0]
            while len(data) < n:
                chunk = s.recv(65536)
                if not chunk:
                    raise ConnectionError("connection closed")
                data += chunk
            buf[0] = data[n:]
            return data[:n]

        def read_reply(idm, deadline):
            while time.time() < deadline:
                b1, b2 = recv_exact(2)
                opcode, length = b1 & 0x0F, b2 & 0x7F
                if length == 126: length = struct.unpack(">H", recv_exact(2))[0]
                elif length == 127: length = struct.unpack(">Q", recv_exact(8))[0]
                if b2 & 0x80: recv_exact(4)
                payload = recv_exact(length)
                if opcode == 8:
                    raise ConnectionError("the robot closed the connection")
                if opcode != 1:
                    continue
                msg = json.loads(payload.decode())
                args = msg.get("args") or {}
                if str(args.get("idm")) != str(idm):
                    continue
                if msg.get("name") == "error":
                    raise RuntimeError(str(args.get("result")))
                return args.get("result")
            raise TimeoutError("no reply within %d s" % timeout)

        deadline = time.time() + timeout
        send({"idm": 1, "params": {"obj": "ServiceDirectory", "member": "service", "args": ["ALSystem"]}})
        service = read_reply(1, deadline)
        if not isinstance(service, dict) or "pyobject" not in service:
            return None, None, "the port speaks web socket but not the RoboHearts protocol (reply: %r)" % (service,)
        send({"idm": 2, "params": {"obj": service["pyobject"], "member": "robotName", "args": []}})
        name = read_reply(2, deadline)
        info = None
        if with_skills:
            try:
                send({"idm": 3, "params": {"obj": "ServiceDirectory", "member": "service", "args": ["PepperSkills"]}})
                skills = read_reply(3, time.time() + timeout)
                send({"idm": 4, "params": {"obj": skills["pyobject"], "member": "info", "args": []}})
                info = read_reply(4, time.time() + 20)
            except (RuntimeError, KeyError, TypeError) as e:
                info = {"error": str(e)}
        try: s.sendall(b"\x88\x80" + os.urandom(4))
        except OSError: pass
        return str(name), info, None
    except (OSError, ssl.SSLError, RuntimeError, ValueError, TimeoutError) as e:
        return None, None, str(e)
    finally:
        try: s.close()
        except OSError: pass

def _bridge_status(host, port, timeout=3.0):
    """The bridge's /status JSON (plain http first, then https with the bridge's own certificate), or None."""
    ctx = ssl.SSLContext(ssl.PROTOCOL_TLS_CLIENT); ctx.check_hostname = False; ctx.verify_mode = ssl.CERT_NONE
    for url, kw in (("http://%s:%d/status" % (host, port), {}), ("https://%s:%d/status" % (host, port), {"context": ctx})):
        try:
            with urllib.request.urlopen(url, timeout=timeout, **kw) as r:
                return json.loads(r.read().decode())
        except Exception:
            continue
    return None

def _is_loopback(host):
    return host in ("127.0.0.1", "localhost", "::1", "[::1]") or host.startswith("127.")

def _host_of(address):
    return str(address or "").strip().split("://")[-1].split("/")[0].split(":")[0]

def check_pepper(robot_ip, via=None):
    """Prints what is reachable and what to do. Returns a dict with 'verdict'.
    robot_ip: Pepper's head address (PEPPER_IP). via: what the notebook connects to (CONNECT_TO), default robot_ip."""
    result = {"verdict": "", "robot_name": None, "details": {}}
    say = print
    if sys.platform == "emscripten":          # JupyterLite (RoboHearts): no sockets in the kernel
        result["verdict"] = "browser"
        say("This kernel runs inside the browser, so it cannot open connections itself.")
        say("Use the connection check in the Step 2 box below instead.")
        return result
    robot = _host_of(robot_ip)
    target = _host_of(via) if via else robot
    if not robot:
        result["verdict"] = "no-address"
        say("PEPPER_IP is empty. Put the address of Pepper's head in it (press the chest button) and run again.")
        return result
    try:
        robot_addr = socket.getaddrinfo(robot, ROBOT_PORT, socket.AF_INET)[0][4][0]
    except socket.gaierror as e:
        result["verdict"] = "bad-address"
        say("'%s' is not an address this computer can look up (%s)." % (robot, e))
        say("Use the numeric address Pepper says when you press its chest button, e.g. 10.219.28.192.")
        return result

    # ---- what Pepper itself answers (always useful context)
    say("Pepper's head at %s:" % robot)
    states = {}
    for port in NAOQI_PORTS + (ROBOT_PORT,):
        states[port], ms = _tcp(robot_addr, port)
        say("  port %-5d %-12s (%d ms)%s" % (port, states[port], ms,
            "  <- RoboHearts service on Pepper (not needed with the bridge)" if port == ROBOT_PORT else "  <- NAOqi"))
    result["details"] = dict(states)
    naoqi_up = any(states[p] == "open" for p in NAOQI_PORTS)
    if naoqi_up:
        cn, err = _tls_common_name(robot_addr, next(p for p in NAOQI_PORTS if states[p] == "open"))
        result["details"]["naoqi_certificate"] = cn or err
        say("  NAOqi certificate: %s  -> Pepper is on and NAOqi is running" % (cn or err))

    # ---- case 1: through the bridge on this computer
    if _is_loopback(target):
        st = _bridge_status(target, ROBOT_PORT)
        result["details"]["bridge"] = st
        if st is None:
            result["verdict"] = "bridge-down"
            say("\nNOT CONNECTED: the Pepper bridge is not running on this computer (nothing answers at http://%s:%d/status)." % (target, ROBOT_PORT))
            say("  Run the Step 1 cell again: it starts bridge/pepper_bridge.py. If it still fails, read bridge/bridge.log next to this notebook.")
            return result
        say("\nBridge on this computer: %s, robot url %s, user %s" % (st.get("bridge"), st.get("robot_url"), st.get("user")))
        if not st.get("connected"):
            result["verdict"] = "bridge-not-logged-in"
            say("NOT CONNECTED: the bridge runs, but it is not connected to Pepper:")
            say("  " + str(st.get("error") or "still connecting; run this cell again in a minute"))
            if naoqi_up:
                say("  Pepper itself is on and NAOqi answers, so the address is right. Check the user name and password:")
                say("  copy bridge/.env.example to bridge/.env, put Pepper's password in PEPPER_PASSWORD (factory default: nao),")
                say("  then run the Step 1 cell again.")
            else:
                say("  Pepper's own NAOqi ports do not answer either: check that Pepper is on, that PEPPER_IP is the head's address")
                say("  (press the chest button) and that this computer is on the same Wi-Fi. Then run the Step 1 cell again.")
            return result
        name, info, err = _ws_robot_name(target, ROBOT_PORT, with_skills=True)
        if name is None:
            result["verdict"] = "bridge-ws-failed"
            say("NOT CONNECTED: the bridge is logged in to Pepper but its web socket did not answer: %s" % err)
            say("  Run the Step 1 cell again (it restarts the bridge if needed).")
            return result
        result.update(verdict="ok", robot_name=name)
        say("\nCONNECTED: Pepper '%s' answers through the bridge (ws://%s:%d -> %s)." % (name, target, ROBOT_PORT, st.get("robot_url")))
        result["details"]["skills"] = info
        if info and "services" in info:
            sv = info["services"]
            yes = lambda k: "available" if sv.get(k) else "NOT available"
            say("  Camera (ALVideoDevice): %s | wheels and arms (ALMotion): %s | speech (ALTextToSpeech): %s"
                % (yes("ALVideoDevice"), yes("ALMotion"), yes("ALTextToSpeech")))
            say("  Forward moves are limited to %s m per command." % info.get("max_forward_m"))
            if not sv.get("ALVideoDevice"):
                say("  Photos from Pepper's camera will not work: choose 'This computer's camera' in Step 2.")
        else:
            say("  The bridge did not report its camera and movement actions (%s):" % ((info or {}).get("error") or "older bridge"))
            say("  run the Step 1 cell again; it restarts the bridge with the current version.")
        say("  Step 2 will connect. If the Step 1 widget above says 'Failed to %s', open https://%s:%d/check in a new tab once," % (target, target, ROBOT_PORT))
        say("  click Advanced, then Proceed (accepts the bridge's certificate), and run the Step 1 cell again.")
        return result

    # ---- case 2: directly to Pepper (RoboHearts' service on port 8443 must run on the robot)
    if states[ROBOT_PORT] == "open":
        name, _info, err = _ws_robot_name(robot_addr, ROBOT_PORT)
        if name is not None:
            result.update(verdict="ok", robot_name=name)
            say("\nCONNECTED: Pepper '%s' answers at wss://%s:%d. Step 1 and Step 2 will connect." % (name, robot, ROBOT_PORT))
        else:
            result["verdict"] = "wrong-service"
            say("\nNOT CONNECTED: something listens on %s:%d but it is not RoboHearts' service: %s" % (robot, ROBOT_PORT, err))
        return result
    if naoqi_up:
        result["verdict"] = "service-down"
        say("\nNOT CONNECTED: Pepper is on and NAOqi is running at %s, but nothing accepts connections on port %d (%s):" % (robot, ROBOT_PORT, states[ROBOT_PORT]))
        say("  the RoboHearts web-socket service that notebooks talk to is not running on Pepper.")
        say("  On this computer, the bridge replaces that service: run the Step 1 cell (it starts bridge/pepper_bridge.py,")
        say("  which logs in to NAOqi at tcps://%s:9503) and then run this cell again." % robot)
        say("  Without the bridge (e.g. on RoboHearts), the service has to be started on the robot: on Pepper's tablet run the")
        say("  'SoftBank application update' and the 'robot applications update' (bottom of the Android settings), then reboot Pepper.")
        return result
    if all(states[p] == "refused" for p in states):
        result["verdict"] = "not-a-pepper"
        say("\nNOT CONNECTED: a device answers at %s but it is not a Pepper head (no NAOqi)." % robot)
        say("  Press Pepper's chest button: the address it says is the head's. The tablet has a different address.")
        return result
    result["verdict"] = "unreachable"
    say("\nNOT CONNECTED: nothing answers at %s." % robot)
    say("  1. Press Pepper's chest button and compare the address it says with PEPPER_IP (the tablet's address is different).")
    say("  2. Make sure this computer and Pepper's head are on the same Wi-Fi (guest and campus networks often separate devices).")
    say("  3. If Pepper has just been switched on, wait two minutes and run this cell again.")
    return result

try:
    _check = check_pepper(PEPPER_IP, via=globals().get("CONNECT_TO"))   # both come from the Step 1 cell
except NameError:
    print("Run the Step 1 cell first: it sets PEPPER_IP. Then run this cell again.")


Pepper's head at 10.219.8.57:
  port 9503  refused      (6 ms)  <- NAOqi
  port 9443  refused      (2 ms)  <- NAOqi
  port 8443  refused      (3 ms)  <- RoboHearts service on Pepper (not needed with the bridge)

Bridge on this computer: pepper-bridge 1.4, robot url tcps://10.219.8.57:9503, user nao
NOT CONNECTED: the bridge runs, but it is not connected to Pepper:
  could not reach Pepper at tcps://10.219.8.57:9503 (Connect error: 61: Connection refused): check that Pepper is on, that PEPPER_IP is the head's address (press the chest button) and that this computer is on the same Wi-Fi.
  Pepper's own NAOqi ports do not answer either: check that Pepper is on, that PEPPER_IP is the head's address
  (press the chest button) and that this computer is on the same Wi-Fi. Then run the Step 1 cell again.


# Step 2 - talk to Pepper through Gemini

Run the cell below once. Everything runs in this browser tab:

1. The **connection check** runs by itself, using the address from Step 1 (the bridge on this computer). A green box means connected. A red box lists what to fix; do that, then press **Check connection**.
2. Press **Wake up Pepper** if Pepper is resting (`ALMotion.wakeUp`, as in the lesson). Moving commands also wake it up by themselves.
3. Paste your **Gemini API key** (kept only in this tab, never saved in the notebook).
4. Press **Start listening**, allow the microphone when the browser asks, say what you want, press **Stop & ask**.
5. Pepper takes a photo with its **head camera** right after you stop, Gemini hears your recording and sees the photo, and Pepper answers. The photo appears under the chat.

What you can say:

| You say | Pepper does |
|---|---|
| *What is this object?* (hold it in front of Pepper's face, about half a metre away) | names the object from the photo and adds a short fact |
| *Pepper, go forward five steps* | rolls forward 5 x 0.2 m = 1 m (`STEP_METERS`), at most 1.5 m per command (`MAX_FORWARD_M`) |
| *Lift right hand* / *Lift left hand* / *Raise both hands* | raises that hand, holds it 3 s (`HAND_HOLD_SECONDS`), lowers it again |
| *Put your hands down* | lowers raised hands |
| any other question | answers it, as before |

**Safety.** Pepper moves only when you clearly ask: a recording without speech is not sent at all, and before a movement a second, independent transcription of your words must contain them (for example *forward* or *steps*, *right* or *left* with *hand* or *arm*, *hands down*); otherwise Pepper asks you to say it again. Pepper's own obstacle protection stays on while it moves. **Stop moving** stops the wheels at once and cancels a hand movement. Untick **Allow Pepper to move** to keep Pepper still (it then only says what it would do). Keep about 1.5 m clear in front of Pepper and close its charging flap.

**Camera.** *Pepper's head camera* is the default; *This computer's camera* uses the laptop webcam instead; *No camera* sends no photo.

**No robot nearby?** Tick *No robot here*: this computer's voice speaks the answers, the laptop webcam takes the photos, and movements are only described in the chat.

Follow-up questions work (the last six exchanges are sent along). **Clear chat** forgets them. Settings such as `STEP_METERS`, `MAX_FORWARD_M`, `HAND_HOLD_SECONDS` and the answer language are at the top of the script in the cell.

In [2]:
from IPython.display import HTML, display
import html, uuid

try:
    PEPPER_ADDRESS = str(CONNECT_TO).strip()  # what Step 1 connected to: the bridge on this computer, or Pepper itself
except NameError:
    try:
        PEPPER_ADDRESS = str(PEPPER_IP).strip()
    except NameError:
        PEPPER_ADDRESS = ""                   # Step 1 has not been run; the page below says so

PAGE = r'''<div id="{{UID}}" class="pv-app" style="font-family:-apple-system,'Segoe UI',Helvetica,Arial,sans-serif;max-width:780px;line-height:1.4;color:#222;background:#fff;padding:10px 12px;border-radius:6px;">
<style>
  .pv-app input[type=text], .pv-app input[type=password] { padding:6px 8px; border:1px solid #bbb; border-radius:4px; font-size:14px; }
  .pv-app button { padding:8px 14px; border:none; border-radius:4px; font-size:14px; cursor:pointer; }
  .pv-app button:disabled { opacity:0.45; cursor:default; }
  .pv-app .pv-row { margin:6px 0; display:flex; align-items:center; gap:8px; flex-wrap:wrap; }
  .pv-app .pv-chat { border:1px solid #ccc; border-radius:4px; padding:8px; min-height:120px; max-height:320px; overflow:auto; background:#fff; }
  .pv-app .pv-diag { border:1px solid #e0b4b4; border-left:5px solid #c62828; border-radius:4px; background:#fff6f6; padding:8px 12px; margin:8px 0; }
  .pv-app .pv-diag.pv-ok { border-color:#b7dfb9; border-left-color:#2a9d3f; background:#f4fbf4; }
  .pv-app .pv-diag ol { margin:6px 0 4px 22px; padding:0; }
  .pv-app .pv-diag li { margin:4px 0; }
  .pv-app .pv-diag .pv-small { color:#666; font-size:12px; margin-top:6px; }
</style>
<h3 style="margin:4px 0 8px;">Talk to Pepper through Gemini</h3>
<div class="pv-row"><label>Pepper address</label><input type="text" class="pv-ip" value="{{PEPPER_IP}}" size="18">
  <button class="pv-check" style="background:#444;color:#fff">Check connection</button>
  <button class="pv-wake" style="background:#1565c0;color:#fff">Wake up Pepper</button>
  <span class="pv-conn" style="color:#b60">Checking ...</span></div>
<div class="pv-diag" style="display:none"></div>
<div class="pv-row"><label>Gemini API key</label><input type="password" class="pv-key" size="42" placeholder="paste your key here">
  <label>Model</label><input type="text" class="pv-model" value="gemini-3.8-flash" size="18"></div>
<div class="pv-row"><label>Camera</label><select class="pv-camera">
    <option value="pepper" selected>Pepper's head camera</option><option value="webcam">This computer's camera</option><option value="off">No camera</option></select>
  <label><input type="checkbox" class="pv-allowmove" checked> Allow Pepper to move (wheels and arms)</label>
  <button class="pv-halt" style="background:#6a1b9a;color:#fff">Stop moving</button></div>
<div class="pv-row"><label><input type="checkbox" class="pv-norobot"> No robot here: use this computer's voice instead of Pepper</label></div>
<div class="pv-row"><button class="pv-start" style="background:#2a9d3f;color:#fff">Start listening</button>
  <button class="pv-stop" style="background:#c62828;color:#fff" disabled>Stop &amp; ask</button>
  <button class="pv-clear" style="background:#ddd;color:#222">Clear chat</button></div>
<div class="pv-status" style="font-weight:600;margin:8px 0;">Checking the connection to Pepper ...</div>
<div class="pv-chat"></div>
<div class="pv-photo" style="display:none;margin:8px 0;"><div style="font-size:12px;color:#666;margin-bottom:4px;">Photo sent to Gemini with the last question:</div>
  <img class="pv-snap" alt="last photo" style="max-width:320px;border:1px solid #ccc;border-radius:4px;"></div>
</div>
<script>
(function () {
  // ---------------------------------------------------------------- settings
  const ANSWER_LANGUAGE = "English";        // must match Pepper's text-to-speech language
  const MAX_ANSWER_WORDS = 60;
  const THINKING_PHRASE = "Let me think.";  // spoken while Gemini works; "" disables it
  const HISTORY_TURNS = 6;
  const SAMPLE_RATE = 16000;
  const ROBOT_PORT = 8443;                  // RoboHearts' secure web socket on Pepper's head (same as ipynao)
  const CONNECT_TIMEOUT_MS = 10000;         // same 10 s wait as ipynao
  const STEP_METERS = 0.2;                  // one "step" when Pepper is asked to go forward (Pepper rolls on wheels)
  const MAX_FORWARD_M = 1.5;                // longest move per command; the bridge enforces the same limit
  const HAND_HOLD_SECONDS = 3;              // a raised hand goes down again after this; 0 keeps it up until "hands down"
  const PEPPER_CAMERA = 0;                  // 0 = camera in Pepper's forehead, 1 = camera in its mouth
  const PHOTO_MAX_SIDE = 640, PHOTO_JPEG_QUALITY = 0.85;   // photo size sent to Gemini
  const MIN_SPEECH_SECONDS = 0.3;           // a recording needs this much speech before anything is sent to Gemini

  // ==== gemini-request: begin (tests/test_gemini_intents.mjs runs this block on its own)
  const ACTIONS = ["none", "move_forward", "raise_right_hand", "raise_left_hand", "raise_both_hands", "lower_hands"];
  const RESPONSE_SCHEMA = { type: "OBJECT", properties: {
      question: { type: "STRING" }, answer: { type: "STRING" },
      action: { type: "STRING", enum: ACTIONS }, steps: { type: "NUMBER" } },
    required: ["question", "answer", "action", "steps"] };
  function systemPrompt(hasPhoto) {
    return `You are Pepper, a friendly humanoid robot on wheels, talking with a person.
The person spoke to you; the audio recording is attached.${hasPhoto ? " A photo from your head camera, taken right after the person finished speaking, is attached too." : " There is no photo this time."}
Reply with JSON only, with these keys:
- "question": a verbatim transcript of what the person said.
- "answer": what you say out loud, in ${ANSWER_LANGUAGE}: plain spoken sentences, no markdown, no lists, no emojis, no special symbols, at most ${MAX_ANSWER_WORDS} words.
- "action": one of ${ACTIONS.map((a) => '"' + a + '"').join(", ")}.
- "steps": the number of steps for "move_forward"; 0 for every other action.
How to choose "action":
- The person asks you to go, move, walk, drive, roll or come forward or ahead: "move_forward". "steps" is the number of steps they asked for. If they give a distance instead, one step is ${STEP_METERS} metres, so one metre is ${Math.round(1 / STEP_METERS)} steps. If they give no number, 1. You move at most ${MAX_FORWARD_M} metres at a time; if they ask for more, still use "move_forward" and say you will go ${MAX_FORWARD_M} metres.
- The person asks you to lift, raise or put up your right hand or arm: "raise_right_hand"; your left one: "raise_left_hand"; both: "raise_both_hands". Right and left are your own. Put your hand or hands down: "lower_hands".
- Anything else is "none". That includes movements you cannot do here, such as going backward, turning, dancing or walking somewhere else: then say kindly which movements you can do (go forward, raise or lower your hands).
- With an action, "answer" says briefly and naturally what you are doing, for example "Okay, moving forward five steps." or "Sure, here is my right hand."
- Choose the action only from what the person says in this recording. Never repeat an earlier action unless this recording clearly asks for it, and never guess a command from unclear audio: then use "none".
${hasPhoto ? `About the photo:
- When the person asks what something is, what they are holding or showing you, what you see, how many there are, what colour it is and so on, answer from the photo: name the object plainly first, then add one short interesting fact about it.
- If the photo does not show it clearly, say so and ask them to hold it in front of your face, about half a metre away.
- Otherwise do not mention the photo.
` : `If the person asks what you see or what they are holding, say that your camera is switched off in the notebook.
`}If the audio contains no understandable speech, set "question" to "", "action" to "none", "steps" to 0 and "answer" to "Sorry, I did not catch that. Could you say it again?".`;
  }
  function buildGeminiBody(wavBase64, photo, history) {
    const parts = [];
    if (history.length) parts.push({ text: "Earlier in this conversation:\n" + history.slice(-HISTORY_TURNS).map((h) => `Person: ${h.q}\nPepper: ${h.a}`).join("\n") + "\n\nNow the person says (audio attached):" });
    parts.push({ inline_data: { mime_type: "audio/wav", data: wavBase64 } });
    if (photo) parts.push({ inline_data: { mime_type: photo.mime, data: photo.data } });
    parts.push({ text: photo ? "Answer the person in the audio. The photo shows what your camera sees now." : "Answer the person in the audio." });
    return { system_instruction: { parts: [{ text: systemPrompt(!!photo) }] }, contents: [{ role: "user", parts }],
      generationConfig: { temperature: 0.2, maxOutputTokens: 500, responseMimeType: "application/json", responseSchema: RESPONSE_SCHEMA, thinkingConfig: { thinkingBudget: 0 } } };
  }
  function readGeminiReply(data) {
    const cand = data.candidates && data.candidates[0];
    const text = cand && cand.content && cand.content.parts ? cand.content.parts.map((p) => p.text || "").join("") : "";
    if (!text) throw new Error("Gemini returned no text: " + JSON.stringify(data).slice(0, 300));
    let parsed;
    try { parsed = JSON.parse(text); }
    catch (e) { throw new Error("Gemini's answer was not complete JSON (finish reason: " + ((cand && cand.finishReason) || "unknown") + "): " + text.slice(0, 120)); }
    const steps = Number(parsed.steps);
    return { question: String(parsed.question || "").trim(), answer: String(parsed.answer || "").trim(),
             action: ACTIONS.includes(parsed.action) ? parsed.action : "none", steps: isFinite(steps) ? steps : 0 };
  }
  // What the robot will do for a reply: {type: "none" | "move_forward" | "raise_hand" | "lower_hands", ...}
  function planAction(reply) {
    switch (reply.action) {
      case "move_forward": {
        const steps = reply.steps > 0 ? reply.steps : 1, wanted = steps * STEP_METERS;
        return { type: "move_forward", steps, meters: Math.round(Math.min(wanted, MAX_FORWARD_M) * 100) / 100, limited: wanted > MAX_FORWARD_M + 1e-9 };
      }
      case "raise_right_hand": return { type: "raise_hand", side: "right" };
      case "raise_left_hand": return { type: "raise_hand", side: "left" };
      case "raise_both_hands": return { type: "raise_hand", side: "both" };
      case "lower_hands": return { type: "lower_hands" };
      default: return { type: "none" };
    }
  }
  function describePlan(p) {
    if (p.type === "move_forward") return "move forward " + p.meters + " m (" + p.steps + " step" + (p.steps === 1 ? "" : "s") + " of " + STEP_METERS + " m" + (p.limited ? ", limited to " + MAX_FORWARD_M + " m" : "") + ")";
    if (p.type === "raise_hand") return p.side === "both" ? "raise both hands" : "raise the " + p.side + " hand";
    if (p.type === "lower_hands") return "lower the hands";
    return "do nothing";
  }
  // Seconds of the recording that sound like speech: 20 ms frames clearly louder than the recording's own background
  // (12 dB above its quietest tenth) and than -45 dBFS. A mouse click, a hum or steady room noise gives (almost) 0.
  function speechSeconds(samples, rate) {
    const frame = Math.max(1, Math.round(rate * 0.02)), rms = [];
    for (let i = 0; i + frame <= samples.length; i += frame) {
      let sum = 0; for (let j = i; j < i + frame; j++) sum += samples[j] * samples[j];
      rms.push(Math.sqrt(sum / frame));
    }
    if (!rms.length) return 0;
    const floor = rms.slice().sort((a, b) => a - b)[Math.floor(rms.length * 0.1)], threshold = Math.max(0.0056, floor * 4);
    return rms.filter((r) => r > threshold).length * frame / rate;
  }
  // A second, neutral transcription (no list of commands in the prompt). A movement runs only if these words ask for it.
  function transcribeBody(wavBase64) {
    return { system_instruction: { parts: [{ text: "Transcribe the human speech in the attached audio verbatim. If the audio contains no clearly audible human speech, return an empty transcript. Never guess or invent words." }] },
      contents: [{ role: "user", parts: [{ inline_data: { mime_type: "audio/wav", data: wavBase64 } }] }],
      generationConfig: { temperature: 0, maxOutputTokens: 300, responseMimeType: "application/json",
        responseSchema: { type: "OBJECT", properties: { transcript: { type: "STRING" } }, required: ["transcript"] }, thinkingConfig: { thinkingBudget: 0 } } };
  }
  function readTranscript(data) {
    const cand = data.candidates && data.candidates[0];
    const text = cand && cand.content && cand.content.parts ? cand.content.parts.map((p) => p.text || "").join("") : "";
    try { return String(JSON.parse(text).transcript || "").trim(); } catch (e) { return ""; }
  }
  // Do these words really ask for the action? (English words; extend the lists for another language.)
  function actionHeard(action, transcript) {
    const t = " " + String(transcript || "").toLowerCase().replace(/[^a-z0-9]+/g, " ") + " ";
    const has = (re) => re.test(t), hand = has(/ (hand|hands|arm|arms) /);
    switch (action) {
      case "move_forward": return has(/ (forward|forwards|ahead|step|steps) /) && !has(/ (back|backward|backwards|backwards) /);
      case "raise_right_hand": return hand && has(/ right /);
      case "raise_left_hand": return hand && has(/ left /);
      case "raise_both_hands": return has(/ (hands|arms) /) || (hand && has(/ both /));
      case "lower_hands": return hand && has(/ (down|lower|drop) /);
      default: return true;
    }
  }
  // ==== gemini-request: end

  // ---------------------------------------------------------------- page elements
  const root = document.getElementById("{{UID}}");
  const $ = (cls) => root.querySelector("." + cls);
  const ipInput = $("pv-ip"), keyInput = $("pv-key"), modelInput = $("pv-model"), noRobot = $("pv-norobot");
  const checkBtn = $("pv-check"), wakeBtn = $("pv-wake"), connLabel = $("pv-conn"), diagEl = $("pv-diag");
  const startBtn = $("pv-start"), stopBtn = $("pv-stop"), clearBtn = $("pv-clear");
  const statusEl = $("pv-status"), chatEl = $("pv-chat");
  const cameraSel = $("pv-camera"), allowMove = $("pv-allowmove"), haltBtn = $("pv-halt"), photoBox = $("pv-photo"), photoImg = $("pv-snap");
  const sleep = (ms) => new Promise((r) => setTimeout(r, ms));
  const errText = (e) => String((e && e.message) || e);
  function setStatus(msg, color) { statusEl.textContent = msg; statusEl.style.color = color || "#222"; }
  function setConn(msg, color) { connLabel.textContent = msg; connLabel.style.color = color; }
  function addChat(who, text, color) {
    const line = document.createElement("div"); line.style.margin = "4px 0";
    const b = document.createElement("b"); b.style.color = color; b.textContent = who + ": ";
    line.appendChild(b); line.appendChild(document.createTextNode(text));
    chatEl.appendChild(line); chatEl.scrollTop = chatEl.scrollHeight;
  }
  function withTimeout(promise, ms, what) {
    return Promise.race([promise, sleep(ms).then(() => { throw new Error(what + " timed out after " + ms / 1000 + " s"); })]);
  }
  try { keyInput.value = sessionStorage.getItem("pv-key") || ""; } catch (e) {}   // the key stays in this tab only
  keyInput.addEventListener("change", () => { try { sessionStorage.setItem("pv-key", keyInput.value); } catch (e) {} });

  // ---------------------------------------------------------------- connection to Pepper
  // Same wire protocol as ipynao 0.8.9's WebsocketQiSession, which the Step 1 widget uses:
  //   browser -> Pepper : {idm, params: {obj, member, args}}
  //   Pepper -> browser : {name: "reply" | "error", args: {idm, result}}
  class QiSession {
    constructor(url) {
      this.url = url; this.pending = {}; this.idm = 0; this.isOpen = false; this.onclosed = null;
      this.socket = new WebSocket(url);
      this.opened = new Promise((resolve, reject) => {
        this.socket.onopen = () => { this.isOpen = true; resolve(this); };
        this.socket.onclose = () => {
          const wasOpen = this.isOpen;
          this.isOpen = false;
          reject(new Error("the connection was refused or closed"));
          for (const id of Object.keys(this.pending)) { this.pending[id].reject("Call " + id + " canceled: disconnected"); delete this.pending[id]; }
          if (wasOpen && this.onclosed) this.onclosed();
        };
      });
      this.opened.catch(() => {});
      this.socket.onmessage = (event) => { let data; try { data = JSON.parse(event.data); } catch (e) { return; } this.handle(data); };
    }
    isConnected() { return this.isOpen; }
    disconnect() { this.onclosed = null; try { this.socket.close(); } catch (e) {} }
    call(obj, member, args) {
      if (!this.isOpen) return Promise.reject(new Error("not connected to Pepper"));
      const idm = ++this.idm;
      const promise = new Promise((resolve, reject) => { this.pending[idm] = { resolve, reject }; });
      this.socket.send(JSON.stringify({ idm, params: { obj, member, args } }));
      return promise;
    }
    service(name) { return this.call("ServiceDirectory", "service", [name]); }
    handle(data) {
      const a = data.args || {}, waiting = this.pending[a.idm];
      if (!waiting) return;
      delete this.pending[a.idm];
      if (data.name === "error") { waiting.reject(a.result); return; }
      const r = a.result;
      if (r !== null && r !== undefined && r.metaobject !== undefined) {   // a service: one function per method
        const service = {}, methods = r.metaobject.methods || {};
        for (const i in methods) { const m = methods[i].name; service[m] = (...args) => this.call(r.pyobject, m, args); }
        waiting.resolve(service);
      } else waiting.resolve(r === undefined ? null : r);
    }
  }

  // Accepts "10.1.2.3", "pepper.local", "10.1.2.3:8443", "https://10.1.2.3:8443/check", with or without spaces.
  function parseAddress(text) {
    let s = String(text || "").trim().replace(/^[a-z][a-z0-9+.-]*:\/\//i, "").replace(/[\/?#].*$/, "");
    let port = ROBOT_PORT;
    const pm = s.match(/^(.*):(\d+)$/); if (pm) { s = pm[1]; port = Number(pm[2]); }
    if (!s) throw new Error("There is no address. Run the Step 1 cell first (it sets PEPPER_IP), then run this cell again.");
    if (!/^[A-Za-z0-9.-]+$/.test(s)) throw new Error('"' + s + '" is not a valid address. It should look like 10.1.2.3 or pepper.local.');
    return { host: s, port, where: s + ":" + port };
  }

  const isLoopback = (h) => /^(127\.|localhost$|\[?::1\]?$)/.test(h);   // the bridge on this computer: plain ws://, no certificate
  async function bridgeStatus(addr) {   // the bridge's /status page; null when no bridge answers
    const ctl = new AbortController(), timer = setTimeout(() => ctl.abort(), 3000);
    try { const r = await fetch("http://" + addr.where + "/status", { cache: "no-store", signal: ctl.signal }); return r.ok ? await r.json() : null; }
    catch (e) { return null; } finally { clearTimeout(timer); }
  }

  // Chrome and Edge ask before a public website may reach a device on the local network
  // (web pages since Chrome 142, web sockets since Chrome 147). Other browsers answer "unknown".
  async function localNetworkPermission() {
    if (!navigator.permissions || !navigator.permissions.query) return "unknown";
    for (const name of ["local-network", "local-network-access"]) {
      try { return (await navigator.permissions.query({ name })).state; } catch (e) {}
    }
    return "unknown";
  }
  function pageIsLocal() {   // a notebook served from this computer or this network is not restricted
    const h = location.hostname;
    return h === "localhost" || h === "[::1]" || /\.local$/.test(h) || /^(127\.|10\.|192\.168\.|169\.254\.|172\.(1[6-9]|2\d|3[01])\.)/.test(h);
  }
  async function probeHttps(url, ms) {   // can this page reach Pepper's secure port at all?
    const started = performance.now(), ctl = new AbortController(), timer = setTimeout(() => ctl.abort(), ms);
    try { await fetch(url, { mode: "no-cors", cache: "no-store", signal: ctl.signal }); return { ok: true, ms: Math.round(performance.now() - started) }; }
    catch (e) { return { ok: false, timedOut: !!(e && e.name === "AbortError"), ms: Math.round(performance.now() - started) }; }
    finally { clearTimeout(timer); }
  }

  // Works out WHY the connection failed and returns what to do about it.
  async function explainFailure(addr) {
    const checkUrl = "https://" + addr.where + "/check";
    const mac = /Mac/i.test(navigator.platform || navigator.userAgent || "");
    const allowSite = "click the icon at the left end of the address bar, open Site settings, set Local network access (some versions say Local network) to Allow, then reload this page and run the cells again.";
    const macStep = "On a Mac: open System Settings > Privacy & Security > Local Network and switch your browser on, then quit and reopen the browser.";
    if (isLoopback(addr.host)) {   // the bridge on this computer
      const st = await bridgeStatus(addr);
      if (!st) return { title: "The Pepper bridge on this computer is not running (nothing answers at " + addr.where + ").",
        steps: ["Run the Step 1 cell again: it starts bridge/pepper_bridge.py and connects it to Pepper.",
                "If it keeps failing, read bridge/bridge.log next to this notebook, or run the check cell under Step 1."],
        details: "no bridge at http://" + addr.where + "/status" };
      if (!st.connected) return { title: "The bridge is running, but it is not connected to Pepper.",
        steps: [String(st.error || "It is still connecting: wait a few seconds and press Check connection."),
                "Check PEPPER_IP in Step 1 (press Pepper's chest button to hear the head's address).",
                "If the message says Pepper closed the connection: the password is wrong. Copy bridge/.env.example to bridge/.env, put Pepper's password in PEPPER_PASSWORD (factory default nao), then run the Step 1 cell again.",
                "Then press Check connection."],
        details: "bridge " + st.bridge + ", robot " + st.robot_url + ", user " + st.user };
      return { title: "The bridge is connected to Pepper" + (st.robot_name ? ' "' + st.robot_name + '"' : "") + ", but this page could not open its web socket.",
        steps: ["Press Check connection again.", "Run the Step 2 cell again; if it still fails, run the Step 1 cell (it restarts the bridge if needed)."],
        details: "bridge " + st.bridge + ", robot " + st.robot_url };
    }
    const perm = await localNetworkPermission();
    if (perm === "denied" && !pageIsLocal()) {
      return { title: "The browser is blocking this site from reaching devices on your local network.",
        steps: ["In Chrome or Edge, " + allowSite],
        note: "Chrome and Edge need this permission since version 147 (April 2026) before a website may connect to a robot on your Wi-Fi. Pepper's own page can still open in a tab while notebooks are blocked. If the setting cannot be changed there, open chrome://settings/content/localNetworkAccess and let sites ask.",
        details: "local network permission: denied" };
    }
    const probe = await probeHttps(checkUrl, 6000);
    const details = "local network permission: " + perm + "; secure page " + (probe.ok ? "answered" : probe.timedOut ? "did not answer" : "failed") + " after " + probe.ms + " ms";
    if (probe.ok) {
      return { title: "Pepper answers at " + addr.where + ", but its robot connection (web socket) did not open or was closed at once.",
        steps: ["Wait a minute and press Check connection again: after start-up Pepper's RoboHearts service needs some time.",
                "Close other notebook tabs that are connected to this Pepper. If the Step 1 widget above is connected, run pepper.disconnect() in a new cell, then press Check connection.",
                "If it keeps failing: on the tablet run the SoftBank application update and the robot applications update (bottom of the Android settings), then reboot Pepper."],
        details };
    }
    if (probe.timedOut) {
      const steps = ["Check the address: press Pepper's chest button and Pepper says the address of its head. The tablet has a different address, which does not work. Put the head's address in PEPPER_IP in Step 1 and run both cells again.",
                     "Check that this computer and Pepper are on the same Wi-Fi. Guest and campus networks often keep devices apart."];
      if (mac) steps.push(macStep);
      steps.push({ text: "To test by hand, open this link in a new tab; a privacy warning or a page means Pepper is reachable: ", link: checkUrl });
      return { title: "Nothing answers at " + addr.where + ".", steps, details };
    }
    const steps = [{ text: "Open this link in a new tab: ", link: checkUrl },
                   "If the tab shows a privacy warning (Your connection is not private): click Advanced, then Proceed. That accepts Pepper's certificate for this address, and every new address needs it once. Come back and press Check connection.",
                   "If the tab says " + addr.host + " refused to connect (ERR_CONNECTION_REFUSED): Pepper is on, but its RoboHearts service, the one notebooks talk to, is not running. On Pepper's tablet run the SoftBank application update and then the robot applications update (bottom of the Android settings), reboot Pepper, wait two minutes after it says its address, then press Check connection. On a computer with the bridge folder next to this notebook, run the Step 1 cell instead: it starts the bridge that replaces this service.",
                   "If the tab says This site can't be reached or took too long to respond: the address is wrong or the network keeps the devices apart. Press Pepper's chest button to hear the address of its head (the tablet has a different one) and check that this computer is on the same Wi-Fi as Pepper." + (mac ? " " + macStep : ""),
                   "If the tab opens normally but this check still fails, the browser is blocking this site. Reload this page, run the cells again and click Allow when the browser asks about devices on your local network. If it does not ask: in Chrome or Edge, " + allowSite];
    const safari = /Safari/.test(navigator.userAgent) && !/Chrome|Chromium|Edg/.test(navigator.userAgent);
    return { title: "This browser could not open a secure connection to " + addr.where + " (its certificate is not accepted yet, or nothing listens on that port).", steps, details,
      note: safari ? "If you are using Safari and it still fails after you accepted the certificate, try Chrome: these lessons were written for it." : "" };
  }
  function showDiagnosis(d) {
    diagEl.innerHTML = ""; diagEl.className = "pv-diag" + (d.ok ? " pv-ok" : ""); diagEl.style.display = "block";
    const title = document.createElement("b"); title.textContent = d.title; diagEl.appendChild(title);
    if (d.steps && d.steps.length) {
      const list = document.createElement("ol");
      for (const step of d.steps) {
        const li = document.createElement("li"), withLink = typeof step === "object";
        li.appendChild(document.createTextNode(withLink ? step.text : step));
        if (withLink) { const a = document.createElement("a"); a.href = step.link; a.textContent = step.link; a.target = "_blank"; a.rel = "noopener"; li.appendChild(a); }
        list.appendChild(li);
      }
      diagEl.appendChild(list);
    }
    if (d.note) { const note = document.createElement("div"); note.textContent = d.note; diagEl.appendChild(note); }
    if (d.details) { const small = document.createElement("div"); small.className = "pv-small"; small.textContent = "Details: " + d.details; diagEl.appendChild(small); }
  }

  const robot = {
    session: null, services: {}, name: "",
    get connected() { return !!this.session && this.session.isConnected(); },
    close() { if (this.session) this.session.disconnect(); this.session = null; this.services = {}; },
    // Opens the web socket. When that fails, the thrown error carries .diagnosis (what to fix).
    async connect(text) {
      const addr = parseAddress(text), url = (isLoopback(addr.host) ? "ws://" : "wss://") + addr.where;
      this.close();
      const perm = await localNetworkPermission();
      const asking = perm === "prompt" && !pageIsLocal();
      if (asking) setStatus("Connecting to Pepper ... If the browser asks whether this site may look for and connect to devices on your local network, click Allow.", "#b60");
      let session;
      try {
        session = new QiSession(url);
        await withTimeout(session.opened, asking ? 20000 : CONNECT_TIMEOUT_MS, "the connection");   // longer while the browser may be asking
      } catch (e) {
        if (session) session.disconnect();
        const err = new Error("could not connect to " + url);
        err.diagnosis = await explainFailure(addr);
        throw err;
      }
      this.session = session; this.services = {}; this.name = "";
      session.onclosed = () => {
        if (this.session !== session) return;
        this.session = null; this.services = {}; setConn("Connection lost", "#b00");
        showDiagnosis({ title: "The connection to Pepper was lost. It reconnects by itself at the next question, or press Check connection." });
      };
      try {   // Pepper's name, to show which robot answered (read only; skipped if the robot does not offer it)
        const sys = await withTimeout(this.service("ALSystem"), 5000, "ALSystem");
        const name = await withTimeout(sys.robotName(), 5000, "robotName");
        if (typeof name === "string") this.name = name;
      } catch (e) {}
      if (!session.isConnected()) {   // Pepper accepted the connection and closed it at once
        this.close();
        const err = new Error("Pepper closed the connection to " + url);
        err.diagnosis = await explainFailure(addr);
        throw err;
      }
      return url;
    },
    async service(name) {
      if (!this.connected) throw new Error("not connected to Pepper (press Check connection)");
      if (!this.services[name]) this.services[name] = await this.session.service(name);
      return this.services[name];
    },
    async say(text) { const s = await this.service("ALAnimatedSpeech"); return s.say(text); },          // speech with gestures
    async sayPlain(text) { const s = await this.service("ALTextToSpeech"); return s.say(text); },       // speech without gestures
    async skills() {   // camera, wheels and arms: the bridge's PepperSkills service (bridge/pepper_bridge.py)
      if (!(await ensureConnected())) throw new Error("not connected to Pepper (see the red box above)");
      try { return await this.service("PepperSkills"); }
      catch (e) { throw new Error("the camera and movements need the Pepper bridge on this computer (run the Step 1 cell): " + errText(e)); }
    },
    async wakeUp() { const s = await this.service("ALMotion"); return s.wakeUp(); },
  };
  // A cell that is run again makes a new copy of this page: close the old copy's connection.
  let disposed = false;
  if (window.__pepperVoice && typeof window.__pepperVoice.close === "function") { try { window.__pepperVoice.close(); } catch (e) {} }
  window.__pepperVoice = { close: () => {
    disposed = true;
    try {
      if (processor) { processor.onaudioprocess = null; processor.disconnect(); }
      if (source) source.disconnect();
      if (stream) stream.getTracks().forEach((t) => t.stop());
      if (audioCtx && audioCtx.state !== "closed") audioCtx.close();
    } catch (e) {}
    try { closeWebcam(); } catch (e) {}
    robot.close();
  } };

  let connecting = null;
  // Connects (or re-uses the open connection). Shows either "Connected" or the reason and the fix.
  function ensureConnected() {
    if (disposed) return Promise.resolve(false);   // this copy was replaced by a newer run of the cell
    if (robot.connected) return Promise.resolve(true);
    if (connecting) return connecting;
    checkBtn.disabled = true; wakeBtn.disabled = true; setConn("Connecting ...", "#b60");
    connecting = (async () => {
      try {
        const url = await robot.connect(ipInput.value);
        setConn("Connected", "#0a0");
        showDiagnosis({ ok: true, title: "Connected to Pepper" + (robot.name ? ' "' + robot.name + '"' : "") + " at " + url + "." });
        return true;
      } catch (e) {
        setConn("Not connected", "#b00");
        showDiagnosis(e.diagnosis || { title: errText(e) });
        return false;
      } finally { checkBtn.disabled = false; wakeBtn.disabled = false; connecting = null; }
    })();
    return connecting;
  }

  // ---------------------------------------------------------------- microphone -> 16 kHz WAV
  let stream = null, audioCtx = null, source = null, processor = null, chunks = [];
  async function startRecording() {
    stream = await navigator.mediaDevices.getUserMedia({ audio: true });
    audioCtx = new (window.AudioContext || window.webkitAudioContext)();
    if (audioCtx.state === "suspended") await audioCtx.resume();
    source = audioCtx.createMediaStreamSource(stream);
    processor = audioCtx.createScriptProcessor(4096, 1, 1);
    chunks = [];
    processor.onaudioprocess = (e) => chunks.push(new Float32Array(e.inputBuffer.getChannelData(0)));
    source.connect(processor); processor.connect(audioCtx.destination);   // output stays silent
    return audioCtx.sampleRate;
  }
  function stopRecording() {
    const rate = audioCtx.sampleRate;
    try { processor.disconnect(); source.disconnect(); } catch (e) {}
    stream.getTracks().forEach((t) => t.stop()); audioCtx.close();
    let total = 0; for (const c of chunks) total += c.length;
    const all = new Float32Array(total); let offset = 0;
    for (const c of chunks) { all.set(c, offset); offset += c.length; }
    const samples = downsample(all, rate, SAMPLE_RATE);
    let peak = 0; for (let i = 0; i < samples.length; i++) { const v = Math.abs(samples[i]); if (v > peak) peak = v; }
    return { wavBase64: base64(encodeWav(samples, SAMPLE_RATE)), seconds: total / rate, peak: Math.round(peak * 32767), speech: speechSeconds(samples, SAMPLE_RATE) };
  }
  function downsample(samples, from, to) {
    if (from === to) return samples;
    const ratio = from / to, outLen = Math.floor(samples.length / ratio), out = new Float32Array(outLen);
    for (let i = 0; i < outLen; i++) {
      const s = Math.floor(i * ratio), e = Math.min(Math.floor((i + 1) * ratio), samples.length);
      let sum = 0; for (let j = s; j < e; j++) sum += samples[j];
      out[i] = e > s ? sum / (e - s) : 0;
    }
    return out;
  }
  function encodeWav(samples, rate) {
    const buf = new ArrayBuffer(44 + samples.length * 2), v = new DataView(buf);
    const str = (o, s) => { for (let i = 0; i < s.length; i++) v.setUint8(o + i, s.charCodeAt(i)); };
    str(0, "RIFF"); v.setUint32(4, 36 + samples.length * 2, true); str(8, "WAVE");
    str(12, "fmt "); v.setUint32(16, 16, true); v.setUint16(20, 1, true); v.setUint16(22, 1, true);
    v.setUint32(24, rate, true); v.setUint32(28, rate * 2, true); v.setUint16(32, 2, true); v.setUint16(34, 16, true);
    str(36, "data"); v.setUint32(40, samples.length * 2, true);
    let o = 44;
    for (let i = 0; i < samples.length; i++, o += 2) { const c = Math.max(-1, Math.min(1, samples[i])); v.setInt16(o, c < 0 ? c * 32768 : c * 32767, true); }
    return buf;
  }
  function base64(buf) {
    const bytes = new Uint8Array(buf); let s = "";
    for (let i = 0; i < bytes.length; i += 32768) s += String.fromCharCode.apply(null, bytes.subarray(i, i + 32768));
    return btoa(s);
  }

  // ---------------------------------------------------------------- photo: Pepper's camera or this computer's camera
  let webcamStream = null, webcamVideo = null;
  function cameraMode() { const m = cameraSel.value; return m === "pepper" && noRobot.checked ? "webcam" : m; }
  async function openWebcam() {
    closeWebcam();
    webcamStream = await navigator.mediaDevices.getUserMedia({ video: { width: 640, height: 480 } });
    webcamVideo = document.createElement("video"); webcamVideo.muted = true; webcamVideo.playsInline = true;
    webcamVideo.srcObject = webcamStream; await webcamVideo.play();
  }
  function closeWebcam() {
    if (webcamStream) webcamStream.getTracks().forEach((t) => t.stop());
    webcamStream = null; webcamVideo = null;
  }
  function loadImage(url) {
    return new Promise((resolve, reject) => { const img = new Image(); img.onload = () => resolve(img); img.onerror = () => reject(new Error("the photo could not be read")); img.src = url; });
  }
  function toJpeg(source, width, height) {   // scaled to PHOTO_MAX_SIDE, as JPEG for a small, fast request
    const scale = Math.min(1, PHOTO_MAX_SIDE / Math.max(width, height)), canvas = document.createElement("canvas");
    canvas.width = Math.round(width * scale); canvas.height = Math.round(height * scale);
    canvas.getContext("2d").drawImage(source, 0, 0, canvas.width, canvas.height);
    const url = canvas.toDataURL("image/jpeg", PHOTO_JPEG_QUALITY);
    return { mime: "image/jpeg", data: url.split(",")[1], url };
  }
  async function takePhoto() {   // null when the camera is off
    const mode = cameraMode();
    if (mode === "off") return null;
    if (mode === "webcam") {
      if (!webcamVideo || !webcamVideo.videoWidth) throw new Error("this computer's camera is not available (allow the camera for this page)");
      return toJpeg(webcamVideo, webcamVideo.videoWidth, webcamVideo.videoHeight);
    }
    const skills = await robot.skills();
    const pic = await withTimeout(skills.takePicture(PEPPER_CAMERA), 15000, "Pepper's camera");
    const img = await loadImage("data:" + pic.mime + ";base64," + pic.data);
    return toJpeg(img, img.naturalWidth, img.naturalHeight);
  }
  function showPhoto(photo) { if (photo) { photoImg.src = photo.url; photoBox.style.display = "block"; } }

  // ---------------------------------------------------------------- Gemini (REST, called from the browser)
  const history = [];
  async function postGemini(body) {
    const key = keyInput.value.trim(), model = modelInput.value.trim();
    if (!key) throw new Error("paste your Gemini API key first");
    const url = "https://generativelanguage.googleapis.com/v1beta/models/" + encodeURIComponent(model) + ":generateContent";
    for (let attempt = 0; ; attempt++) {
      const resp = await fetch(url, { method: "POST", headers: { "Content-Type": "application/json", "x-goog-api-key": key }, body: JSON.stringify(body) });
      if ((resp.status === 429 || resp.status === 503) && attempt < 2) { setStatus("Gemini is busy (" + resp.status + "), retrying ...", "#b60"); await sleep(2000 * (attempt + 1)); continue; }
      if (!resp.ok) {
        const detail = (await resp.text()).slice(0, 300);
        // models that only work with thinking refuse "no thinking": ask again without it, with room for the thinking
        if (resp.status === 400 && /thinking/i.test(detail) && body.generationConfig.thinkingConfig) {
          delete body.generationConfig.thinkingConfig; body.generationConfig.maxOutputTokens = 4096; continue;
        }
        throw new Error("Gemini HTTP " + resp.status + ": " + detail);
      }
      return resp.json();
    }
  }
  async function askGemini(wavBase64, photo) {
    const started = Date.now();
    const reply = readGeminiReply(await postGemini(buildGeminiBody(wavBase64, photo, history)));
    reply.seconds = (Date.now() - started) / 1000;
    return reply;
  }
  async function transcribe(wavBase64) { return readTranscript(await postGemini(transcribeBody(wavBase64))); }

  // ---------------------------------------------------------------- speaking
  function cleanForTts(text) { return text.replace(/[\\^]/g, " ").replace(/\s+/g, " ").trim(); }   // NAOqi tag characters
  function browserSay(text) {
    return new Promise((resolve) => {
      if (!("speechSynthesis" in window)) { console.log("Pepper would say:", text); resolve(); return; }
      const u = new SpeechSynthesisUtterance(text); u.lang = "en-US";
      u.onend = () => resolve(); u.onerror = () => resolve();
      window.speechSynthesis.cancel(); window.speechSynthesis.speak(u);
      setTimeout(resolve, 3000 + 400 * text.split(/\s+/).length);   // in case onend never fires
    });
  }
  async function speakPlain(text) {   // no gestures: used next to movements, so speech and arm commands never collide
    const clean = cleanForTts(text);
    if (noRobot.checked) return browserSay(clean);
    if (!(await ensureConnected())) throw new Error("not connected to Pepper (see the red box above, or tick 'No robot here')");
    return robot.sayPlain(clean);
  }
  async function speak(text) {
    const clean = cleanForTts(text);
    if (noRobot.checked) return browserSay(clean);
    if (!(await ensureConnected())) throw new Error("not connected to Pepper (see the red box above, or tick 'No robot here')");
    return robot.say(clean);
  }

  // ---------------------------------------------------------------- buttons
  async function checkConnection() {
    if (noRobot.checked) { setConn("No robot", "#666"); diagEl.style.display = "none"; setStatus("No robot: this computer's voice answers. Press Start listening and ask your question."); return; }
    setStatus("Checking the connection to Pepper ...", "#b60");
    if (await ensureConnected()) {
      setConn("Connected", "#0a0");
      if (diagEl.style.display === "none") showDiagnosis({ ok: true, title: "Connected to Pepper" + (robot.name ? ' "' + robot.name + '"' : "") + "." });
      setStatus("Connected. Press Wake up Pepper if Pepper is resting, then Start listening and ask your question.");
    }
    else setStatus("Not connected to Pepper yet. Follow the steps in the red box, then press Check connection.", "#b00");
  }
  checkBtn.onclick = () => { robot.close(); checkConnection(); };   // always a fresh attempt
  ipInput.addEventListener("change", () => { robot.close(); setConn("Not connected", "#b00"); });
  noRobot.addEventListener("change", checkConnection);
  wakeBtn.onclick = async () => {
    if (!(await ensureConnected())) { setStatus("Not connected to Pepper yet. Follow the steps in the red box, then press Check connection.", "#b00"); return; }
    wakeBtn.disabled = true; setStatus("Waking Pepper up ...", "#06c");
    try { await withTimeout(robot.wakeUp(), 30000, "wakeUp"); setStatus("Pepper's motors are on. Press Start listening and ask your question."); }
    catch (e) { setStatus("ALMotion.wakeUp failed: " + errText(e), "#b60"); }
    wakeBtn.disabled = false;
  };
  startBtn.onclick = async () => {
    startBtn.disabled = true;
    let rate, camNote = "";
    if (cameraMode() === "webcam") { try { await openWebcam(); } catch (e) { camNote = " (this computer's camera is off: " + errText(e) + ")"; } }
    try { rate = await startRecording(); }
    catch (e) { closeWebcam(); setStatus("Microphone error: " + errText(e) + " (allow the microphone for this page)", "#b00"); startBtn.disabled = false; return; }
    stopBtn.disabled = false; cameraSel.disabled = true; noRobot.disabled = true;
    setStatus("Listening (" + rate + " Hz). " + (cameraMode() === "off" ? "" : "Show the object to " + (cameraMode() === "pepper" ? "Pepper's face" : "this computer's camera") + " if you ask about one. ") + "Ask your question, then press Stop & ask." + camNote, "#0a0");
  };

  // Runs the action Gemini chose, while Pepper says the answer (speech without gestures, so it cannot fight the arms).
  async function perform(plan) {
    const skills = await robot.skills();
    if (plan.type === "move_forward") return withTimeout(skills.moveForward(plan.meters), 60000, "moving forward");
    if (plan.type === "raise_hand") return withTimeout(skills.raiseHand(plan.side, HAND_HOLD_SECONDS), 30000 + 1000 * HAND_HOLD_SECONDS, "raising the hand");
    if (plan.type === "lower_hands") return withTimeout(skills.lowerHands(), 30000, "lowering the hands");
    return null;
  }
  function doneText(plan, done) {
    if (plan.type === "move_forward") {
      if (done.reached === false) {
        const moved = typeof done.moved === "number" ? done.moved : null;
        let t = moved !== null && moved < 0.05 ? "did not move" : "moved only " + (moved !== null ? moved + " m" : "part of the way");
        if (done.reason === "Safety") t += ": Pepper's own safety system saw an obstacle" + (done.obstacle_m ? " about " + done.obstacle_m + " m ahead" : "") + " and refused. Clear the space in front of Pepper (about 1.5 m, also low things like chair legs and table edges) and ask again";
        else if (done.reason === "WheelsDisabled") t += ": Pepper keeps its wheels switched off. Close the charging flap at the back of its base (the small hatch where the charger plugs in), make sure nothing is plugged in, then ask again";
        else if (done.reason) t += ": Pepper stopped the move itself (" + done.reason + "). Press Wake up Pepper and ask again";
        else t += " (an obstacle, or Stop moving was pressed)";
        return t;
      }
      let t = "moved forward " + done.meters + " m";
      if (done.limited || plan.limited) t += " (limited to " + done.meters + " m)";
      return t;
    }
    if (plan.type === "raise_hand") return (plan.side === "both" ? "raised both hands" : "raised the " + plan.side + " hand")
      + (done && done.lowered ? " and lowered it again after " + done.held_s + " s" : done && done.stopped ? " (Stop moving: it stays up; say \"put your hands down\")" : "");
    return "lowered the hands";
  }
  async function act(plan, answer, note) {
    note = note || "";
    const what = describePlan(plan);
    if (noRobot.checked) {
      await browserSay(cleanForTts(answer));
      addChat("Pepper", "(no robot: Pepper would " + what + ")", "#666");
      setStatus("No robot: Pepper would " + what + ". Press Start listening to ask another question.");
      return;
    }
    if (!allowMove.checked) {
      try { await withTimeout(speakPlain("Moving is switched off in the notebook."), 30000, "Pepper"); } catch (e) {}
      addChat("Pepper", "(movement is switched off: did not " + what + ")", "#b60");
      setStatus("Movement is switched off. Tick 'Allow Pepper to move' to let Pepper " + what + ".", "#b60");
      return;
    }
    setStatus("Pepper is going to " + what + " ...", "#06c");
    const speech = withTimeout(speakPlain(answer), 60000, "Pepper").then(() => null, (e) => e);
    let done = null, failed = null;
    try { done = await perform(plan); } catch (e) { failed = e; }
    const spoke = await speech;
    if (failed) {
      addChat("Pepper", "(could not " + what + ": " + errText(failed) + ")", "#b00");
      setStatus("Pepper could not " + what + ": " + errText(failed) + note, "#b00");
      return;
    }
    addChat("Pepper", "(" + doneText(plan, done) + ")", "#0a0");
    setStatus("Done: Pepper " + doneText(plan, done) + "." + (spoke ? " (speech: " + errText(spoke) + ")" : "") + note + " Press Start listening to ask another question.");
  }

  stopBtn.onclick = async () => {
    stopBtn.disabled = true;
    let rec;
    try { rec = stopRecording(); } catch (e) { setStatus("Microphone error: " + errText(e), "#b00"); closeWebcam(); startBtn.disabled = false; cameraSel.disabled = false; noRobot.disabled = false; return; }
    try {
      if (rec.seconds < 0.5) { setStatus("The recording was too short. Press Start listening and ask again.", "#b00"); return; }
      if (rec.peak < 200) { setStatus("Only silence was recorded. Check the browser's microphone permission for this page.", "#b00"); return; }
      if (rec.speech < MIN_SPEECH_SECONDS) { setStatus("No speech was heard (" + rec.speech.toFixed(2) + " s). Nothing was sent. Press Start listening, speak, then press Stop & ask.", "#b00"); return; }
      photoBox.style.display = "none"; photoImg.removeAttribute("src");
      const thinking = THINKING_PHRASE ? withTimeout(speakPlain(THINKING_PHRASE), 30000, "Pepper").then(() => "", (e) => " (Pepper warning: " + errText(e) + ")") : Promise.resolve("");
      let photo = null, photoNote = "";
      if (cameraMode() !== "off") {
        setStatus("Recorded " + rec.seconds.toFixed(1) + " s. Taking a photo ...", "#06c");
        try { photo = await takePhoto(); showPhoto(photo); } catch (e) { photoNote = " (no photo: " + errText(e) + ")"; }
      }
      closeWebcam();
      setStatus("Asking Gemini (" + modelInput.value + ")" + (photo ? " with the photo" : "") + " ..." + photoNote, "#06c");
      let result;
      try { result = await askGemini(rec.wavBase64, photo); } catch (e) { setStatus("Gemini error: " + errText(e) + photoNote, "#b00"); return; }
      const warning = (await thinking) + photoNote;
      if (result.action !== "none") {   // a movement: a second, neutral transcription must contain the words for it
        setStatus("Checking that you asked Pepper to " + describePlan(planAction(result)) + " ...", "#06c");
        let heard = null;
        try { heard = await withTimeout(transcribe(rec.wavBase64), 20000, "the check"); } catch (e) { heard = null; }
        if (heard === null || !actionHeard(result.action, heard)) {
          addChat("You", heard || "(nothing clear)", "#06c");
          const sorry = "Sorry, I am not sure you asked me to move. Please say it again, for example: go forward five steps, or lift your right hand.";
          addChat("Pepper", sorry, "#0a0");
          addChat("Pepper", "(did not " + describePlan(planAction(result)) + ": " + (heard === null ? "the check transcription failed" : "the check transcription heard \"" + heard + "\"") + ")", "#b60");
          setStatus("Pepper did not move: the words did not clearly ask for it. Say the movement again." + warning, "#b60");
          try { await withTimeout(speakPlain(sorry), 60000, "Pepper"); } catch (e) {}
          return;
        }
      }
      addChat("You", result.question || "(no speech detected)", "#06c");
      addChat("Pepper", result.answer, "#0a0");
      if (result.question) history.push({ q: result.question, a: result.answer });
      const plan = planAction(result);
      if (plan.type !== "none") { await act(plan, result.answer, warning); return; }
      setStatus("Gemini answered in " + result.seconds.toFixed(1) + " s. Pepper is speaking ..." + warning, "#0a0");
      try { await withTimeout(speak(result.answer), 120000, "Pepper"); setStatus("Done. Press Start listening to ask another question." + warning); }
      catch (e) { setStatus("Pepper could not speak: " + errText(e), "#b00"); }
    } finally { closeWebcam(); startBtn.disabled = false; cameraSel.disabled = false; noRobot.disabled = false; }
  };
  haltBtn.onclick = async () => {   // stops the wheels at once
    if (noRobot.checked) return;
    try { const s = await robot.skills(); await withTimeout(s.stop(), 10000, "stop"); setStatus("Stopped: wheels stopped, a hand movement under way is cancelled.", "#b00"); }
    catch (e) { setStatus("Stop moving failed: " + errText(e), "#b00"); }
  };
  clearBtn.onclick = () => { history.length = 0; chatEl.innerHTML = ""; setStatus("Chat cleared. Press Start listening to ask a question."); };

  checkConnection();   // runs by itself when the cell is run; it only opens the connection, Pepper does not move
})();
</script>'''

display(HTML(PAGE.replace("{{UID}}", "pv-" + uuid.uuid4().hex[:8]).replace("{{PEPPER_IP}}", html.escape(PEPPER_ADDRESS, quote=True))))

# If something does not work

**Pepper does not connect** (Step 1 says *Failed*, Step 2 shows a red box). Run the check cell under Step 1: it names the cause and the fix. The usual causes:

* **The bridge is not running on this computer.** Run the Step 1 cell again: it starts `bridge/pepper_bridge.py` and prints what happened; the log is `bridge/bridge.log`. If the cell says `bridge/.venv is missing`, run `bash bridge/setup.sh` in a terminal once (it installs Python 3.12 and the `qi` package, which is NAOqi's official Python library).
* **The bridge runs, but "Pepper closed the connection".** The NAOqi user name or password is wrong. Copy `bridge/.env.example` to `bridge/.env`, put Pepper's password in `PEPPER_PASSWORD` (user `nao`; the factory default password is `nao`), and run the Step 1 cell again.
* **Step 1 says *Failed to 127.0.0.1* while the check cell says CONNECTED.** The browser has not accepted the bridge's certificate: open `https://127.0.0.1:8443/check` in a new tab, click *Advanced*, then *Proceed*, and run the Step 1 cell again. Needed once per browser session; Step 2 does not need it.
* **Pepper is off, the address is not the head's, or the network keeps the devices apart.** The check cell says *nothing answers*. Press the chest button to hear the head's address (the tablet has a different one). Computer and Pepper must be on the same Wi-Fi. Wait two minutes after switching Pepper on.
* **On RoboHearts (no bridge): Pepper's own RoboHearts service is not running** (port 8443 refused while NAOqi's ports answer). On Pepper's tablet run the *SoftBank application update*, then the *robot applications update* (bottom of the Android settings), reboot Pepper and wait two minutes. RoboHearts' support page notes that these services may not start at the very first install.
* **The browser blocks the site from your local network** (RoboHearts only). Chrome and Edge ask whether the site may "look for and connect to any device on your local network". If that question was blocked or closed, click the icon at the left end of the address bar, open *Site settings*, set *Local network access* to *Allow*, and reload the page.
* **Pepper's certificate is not accepted** (RoboHearts only). Open `https://<PEPPER_IP>:8443/check` in a new tab, click *Advanced*, then *Proceed*. Every new address needs this once.

**Pepper does not move or does not take photos**

* **"the camera and movements need the Pepper bridge"** - run the Step 1 cell first: the photos and movements go through the bridge on this computer.
* **"Pepper is still doing the previous movement"** - one movement at a time; ask again when it has finished.
* **"Pepper did not move: the words did not clearly ask for it"** - say the movement plainly, for example *go forward five steps*, *lift your right hand*, *put your hands down*. Phrases such as *do that again* are not enough on purpose.
* **"No speech was heard"** - nothing was sent to Gemini; speak a little louder or closer to the microphone.
* **"did not move: Pepper's own safety system saw an obstacle about 0.5 m ahead"** - Pepper refuses to roll towards anything within about half a metre, including low things its cameras see (chair legs, table edges, bags). Give it 1.5 m of clear floor ahead and ask again. **"Pepper keeps its wheels switched off"** means the robot has locked its base: this happens while the charging flap at the back of its base is open or the charger is plugged in; close the flap and ask again. "Pepper stopped the move itself" means another behaviour interrupted it: press **Wake up Pepper** and try again.
* **"could not move" with a NAOqi error** - press **Wake up Pepper** and try again; the error text says what NAOqi refused.
* **The answer about an object is wrong or vague** - hold the object in front of Pepper's face, about half a metre away, in good light; the photo under the chat shows what Gemini saw. *This computer's camera* is an alternative.

Other messages:

* **"Microphone error"** - allow the microphone for this page in the browser's site permissions, then press Start listening again.
* **"Only silence was recorded"** - the browser used a muted or wrong microphone; check the browser's microphone choice for this site.
* **Gemini HTTP 400/403** - the API key is wrong or the model name is not available to your key. Google retires model names over time; `gemini-flash-latest` is the moving alias.
* **Pepper answers in the wrong language** - `ANSWER_LANGUAGE` at the top of the script must match Pepper's text-to-speech language.
* **The cell was run twice** - that is fine: the newest copy closes the older copy's connection. Use the newest one.